# Person 3 Evaluation: MobileNetV3-Large

Notebook สำหรับทีม Evaluation โดยเฉพาะ ไม่ฝึกโมเดลและไม่ต้องใช้ชุด Train

สิ่งที่ต้องมี: checkpoint `.pt` และชุด Test แบบ ZIP หรือโฟลเดอร์ หลังตั้ง path แล้วสามารถกด **Run all** ได้

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi


## 1. ตั้งค่า Path

แก้เฉพาะ path ของชุด Test หากทีมเก็บไฟล์ไว้ตำแหน่งอื่น รองรับทั้ง ZIP และโฟลเดอร์

In [ ]:
from pathlib import Path

MODEL_PATH = Path('/content/drive/MyDrive/dl/thai_char_person3_outputs/mobilenetv3_large_best.pt')
LABELS_PATH = Path('/content/drive/MyDrive/dl/person3_evaluation/labels.csv')

# ใช้อย่างใดอย่างหนึ่ง: ZIP หรือโฟลเดอร์
TEST_ZIP_PATH = Path('/content/drive/MyDrive/dl/TH_Characters_Test_5pct.zip')
TEST_FOLDER_PATH = Path('/content/drive/MyDrive/dl/TH_Characters_Test_5pct')

# ปล่อยเป็น None เพื่อค้นหา ground_truth.csv ในชุด Test อัตโนมัติ
GROUND_TRUTH_PATH = None
OUTPUT_DIR = Path('/content/drive/MyDrive/dl/evaluation_results/person3_mobilenetv3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_PATH.exists(), f'ไม่พบ checkpoint: {MODEL_PATH}'
print('Model:', MODEL_PATH)
print('Results:', OUTPUT_DIR)


## 2. เตรียม Runtime และชุด Test

In [ ]:
import hashlib
import json
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from google.colab import files
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')

LOCAL_TEST_DIR = Path('/content/person3_test')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

if TEST_ZIP_PATH.exists():
    print('พบ Test ZIP ใน Google Drive:', TEST_ZIP_PATH)
elif TEST_FOLDER_PATH.exists():
    print('พบ Test folder ใน Google Drive:', TEST_FOLDER_PATH)
else:
    print('ไม่พบชุด Test ใน Drive กรุณาเลือกไฟล์ ZIP จากเครื่อง')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('ยังไม่ได้อัปโหลดชุด Test')
    uploaded_name = next(iter(uploaded))
    TEST_ZIP_PATH = Path('/content') / uploaded_name
    if TEST_ZIP_PATH.suffix.lower() != '.zip':
        raise ValueError(f'กรุณาอัปโหลดไฟล์ .zip: {TEST_ZIP_PATH.name}')

if TEST_ZIP_PATH.exists():
    if not LOCAL_TEST_DIR.exists():
        print('กำลังแตก Test ZIP ลง local SSD ของ Colab...')
        shutil.unpack_archive(TEST_ZIP_PATH, LOCAL_TEST_DIR)
    TEST_ROOT = LOCAL_TEST_DIR
elif TEST_FOLDER_PATH.exists():
    TEST_ROOT = TEST_FOLDER_PATH
else:
    raise FileNotFoundError('ไม่พบทั้ง TEST_ZIP_PATH และ TEST_FOLDER_PATH')

image_files = sorted(
    path for path in TEST_ROOT.rglob('*')
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
if not image_files:
    raise RuntimeError(f'ไม่พบภาพภายใน {TEST_ROOT}')

print(f'Test root: {TEST_ROOT}')
print(f'Images: {len(image_files):,}')

## 3. โหลด Checkpoint และสร้าง Model

Class order อ่านจาก checkpoint โดยตรง จึงไม่ต้องเดาลำดับโฟลเดอร์

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
required_keys = {'model_state_dict', 'class_to_idx', 'image_size', 'best_val_acc'}
missing_keys = required_keys.difference(checkpoint)
if missing_keys:
    raise KeyError(f'Checkpoint ขาด keys: {sorted(missing_keys)}')

class_to_idx = {str(code): int(index) for code, index in checkpoint['class_to_idx'].items()}
num_classes = len(class_to_idx)
idx_to_code = [None] * num_classes
for code, index in class_to_idx.items():
    idx_to_code[index] = code
if any(code is None for code in idx_to_code):
    raise ValueError('class_to_idx ไม่ต่อเนื่อง')

image_size = int(checkpoint['image_size'])
model = models.mobilenet_v3_large(weights=None)
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device).eval()

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

model_sha256 = sha256_file(MODEL_PATH)
print('Architecture: MobileNetV3-Large')
print('Classes:', num_classes)
print('Input size:', image_size)
print(f"Checkpoint validation accuracy: {float(checkpoint['best_val_acc']):.2%}")
print('SHA256:', model_sha256)


## 4. Preprocessing Contract

ใช้ RGB, white letterbox และ ImageNet normalization เหมือนตอนฝึกทุกประการ

In [ ]:
class Letterbox:
    def __init__(self, size, fill=(255, 255, 255)):
        self.size = (size, size)
        self.fill = fill

    def __call__(self, image):
        image = image.convert('RGB')
        return ImageOps.pad(
            image, self.size, method=Image.Resampling.BILINEAR,
            color=self.fill, centering=(0.5, 0.5),
        )

eval_transform = transforms.Compose([
    Letterbox(image_size),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

labels_df = pd.read_csv(LABELS_PATH) if LABELS_PATH.exists() else None
if labels_df is not None:
    labels_df['tis620'] = labels_df['tis620'].astype(str)
    char_by_code = dict(zip(labels_df.tis620, labels_df.char))
    name_by_code = dict(zip(labels_df.tis620, labels_df.name_en))
else:
    char_by_code = {}
    name_by_code = {}
    print('หมายเหตุ: ไม่พบ labels.csv จะแสดงผลด้วยรหัสคลาสแทนตัวอักษร')


## 5. Inference

สร้าง `predictions.csv` เสมอ แม้ชุด Test ไม่มีเฉลย

In [ ]:
class EvaluationImages(Dataset):
    def __init__(self, root, files, transform):
        self.root = Path(root)
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]
        with Image.open(path) as image:
            tensor = self.transform(image)
        return tensor, path.relative_to(self.root).as_posix()

BATCH_SIZE = 128
NUM_WORKERS = 2
test_dataset = EvaluationImages(TEST_ROOT, image_files, eval_transform)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda',
    persistent_workers=NUM_WORKERS > 0,
)

rows = []
started = time.perf_counter()
with torch.inference_mode():
    for inputs, filenames in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'
        ):
            probabilities = model(inputs).softmax(dim=1).cpu()
        indices = probabilities.argmax(dim=1).numpy()
        confidences = probabilities.max(dim=1).values.numpy()
        for filename, index, confidence in zip(filenames, indices, confidences):
            code = idx_to_code[int(index)]
            rows.append({
                'filename': filename,
                'basename': Path(filename).name,
                'predicted_index': int(index),
                'predicted_class': code,
                'predicted_char': char_by_code.get(code, ''),
                'predicted_name_en': name_by_code.get(code, ''),
                'confidence': float(confidence),
            })
elapsed_seconds = time.perf_counter() - started
predictions_df = pd.DataFrame(rows)
predictions_df.to_csv(
    OUTPUT_DIR / 'predictions.csv', index=False, encoding='utf-8-sig'
)
print(f'Predicted: {len(predictions_df):,} images')
print(f'Inference time: {elapsed_seconds:.2f} seconds')
print(f'Throughput: {len(predictions_df) / elapsed_seconds:.1f} images/second')
display(predictions_df.head())


## 6. Ground Truth Detection

รองรับ `ground_truth.csv` ที่มีคอลัมน์ชื่อไฟล์และ class รวมถึงชุด Test ที่แบ่งภาพเป็น `<class>/image.jpg` หากไม่มีเฉลยจะจบที่ predictions.csv โดยไม่เกิด error

In [ ]:
def normalize_class(value):
    text = str(value).strip()
    if text.endswith('.0'):
        text = text[:-2]
    if text in class_to_idx:
        return text
    try:
        index = int(text)
    except ValueError:
        return text
    return idx_to_code[index] if 0 <= index < num_classes else text

truth_path = Path(GROUND_TRUTH_PATH) if GROUND_TRUTH_PATH else None
if truth_path is None:
    candidates = list(TEST_ROOT.rglob('ground_truth.csv'))
    truth_path = candidates[0] if candidates else None

scored_df = None
if truth_path is not None and truth_path.exists():
    truth_df = pd.read_csv(truth_path)
    filename_column = next(
        (column for column in ['filename', 'file', 'path', 'image'] if column in truth_df.columns),
        None,
    )
    label_column = next(
        (column for column in ['class', 'tis620', 'label', 'true_class', 'class_id'] if column in truth_df.columns),
        None,
    )
    if filename_column is None or label_column is None:
        raise ValueError(f'ไม่พบคอลัมน์ filename/class ใน {truth_path}')
    truth_df = truth_df.copy()
    truth_df['basename'] = truth_df[filename_column].map(lambda value: Path(str(value)).name)
    truth_df['true_class'] = truth_df[label_column].map(normalize_class)
    if truth_df.basename.duplicated().any() or predictions_df.basename.duplicated().any():
        raise ValueError('basename ซ้ำ กรุณาใช้ ground truth ที่มีชื่อไฟล์ไม่ซ้ำ')
    scored_df = truth_df[['basename', 'true_class']].merge(
        predictions_df, on='basename', how='inner', validate='one_to_one'
    )
    print('Ground truth:', truth_path)
elif predictions_df.filename.map(lambda value: Path(value).parent.name in class_to_idx).all():
    scored_df = predictions_df.copy()
    scored_df['true_class'] = scored_df.filename.map(lambda value: Path(value).parent.name)
    print('Ground truth: inferred from class folders')
else:
    print('ไม่พบ ground truth: สร้าง predictions.csv แล้ว แต่จะไม่คำนวณ metrics')

if scored_df is not None:
    if len(scored_df) != len(predictions_df):
        raise RuntimeError(
            f'จับคู่เฉลยได้ {len(scored_df):,}/{len(predictions_df):,} ภาพ'
        )
    invalid = set(scored_df.true_class).difference(class_to_idx)
    if invalid:
        raise ValueError(f'พบคลาสที่ checkpoint ไม่รู้จัก: {sorted(invalid)}')


## 7. Metrics และ Confusion Matrix

Cell นี้ทำงานเมื่อพบ ground truth เท่านั้น

In [ ]:
if scored_df is None:
    print('Skipped metrics: test set ไม่มี ground truth')
else:
    y_true_code = scored_df.true_class.astype(str).to_numpy()
    y_pred_code = scored_df.predicted_class.astype(str).to_numpy()
    y_true = np.array([class_to_idx[code] for code in y_true_code])
    y_pred = np.array([class_to_idx[code] for code in y_pred_code])

    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')
    print(f'Accuracy:          {accuracy:.2%}')
    print(f'Balanced Accuracy: {balanced_accuracy:.2%}')
    print(f'Macro F1:          {macro_f1:.4f}')
    print(f'Weighted F1:       {weighted_f1:.4f}')

    display_names = [
        f"{char_by_code.get(code, '')} ({code})" if char_by_code.get(code) else code
        for code in idx_to_code
    ]
    report = classification_report(
        y_true, y_pred, labels=np.arange(num_classes),
        target_names=display_names, output_dict=True, zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    report_df.to_csv(OUTPUT_DIR / 'classification_report.csv', encoding='utf-8-sig')
    scored_df['correct'] = y_true == y_pred
    scored_df.to_csv(OUTPUT_DIR / 'scored_predictions.csv', index=False, encoding='utf-8-sig')

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
    plt.figure(figsize=(24, 20))
    sns.heatmap(cm, cmap='Blues', xticklabels=idx_to_code, yticklabels=idx_to_code)
    plt.title(
        f'MobileNetV3-Large | Accuracy {accuracy:.2%} | Macro F1 {macro_f1:.4f}'
    )
    plt.xlabel('Predicted class')
    plt.ylabel('True class')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=180, bbox_inches='tight')
    plt.show()

    errors = cm.copy()
    np.fill_diagonal(errors, 0)
    pairs = []
    for flat_index in np.argsort(errors.ravel())[::-1][:20]:
        true_index, pred_index = np.unravel_index(flat_index, errors.shape)
        count = int(errors[true_index, pred_index])
        if count == 0:
            break
        pairs.append({
            'true_class': idx_to_code[true_index],
            'predicted_class': idx_to_code[pred_index],
            'count': count,
        })
    top_confusions_df = pd.DataFrame(pairs)
    top_confusions_df.to_csv(OUTPUT_DIR / 'top_confusions.csv', index=False)
    display(report_df.tail(5))
    display(top_confusions_df)


## 8. Evaluation Summary

บันทึก metadata, hash ของ checkpoint, เวลา inference และ metrics เพื่อให้เทียบกับโมเดลอื่นได้อย่างตรวจสอบย้อนกลับได้

In [ ]:
summary = {
    'model_name': 'MobileNetV3-Large',
    'checkpoint': MODEL_PATH.name,
    'checkpoint_sha256': model_sha256,
    'checkpoint_validation_accuracy': float(checkpoint['best_val_acc']),
    'num_classes': num_classes,
    'image_size': image_size,
    'test_images': len(predictions_df),
    'model_size_mb': MODEL_PATH.stat().st_size / 1024**2,
    'inference_seconds': elapsed_seconds,
    'images_per_second': len(predictions_df) / elapsed_seconds,
    'accuracy': float(accuracy) if scored_df is not None else None,
    'balanced_accuracy': float(balanced_accuracy) if scored_df is not None else None,
    'macro_f1': float(macro_f1) if scored_df is not None else None,
    'weighted_f1': float(weighted_f1) if scored_df is not None else None,
}
with open(OUTPUT_DIR / 'evaluation_summary.json', 'w', encoding='utf-8') as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)
display(pd.Series(summary))
print('Saved all results to:', OUTPUT_DIR)


## ไฟล์ผลลัพธ์

- `predictions.csv`: ผลทำนายทุกภาพ (สร้างเสมอ)
- `evaluation_summary.json`: metadata และ metrics สรุป
- `classification_report.csv`: precision/recall/F1 รายคลาส เมื่อมีเฉลย
- `scored_predictions.csv`: ผลรายภาพพร้อม true class เมื่อมีเฉลย
- `confusion_matrix.png`: confusion matrix เมื่อมีเฉลย
- `top_confusions.csv`: คู่คลาสที่สับสนสูงสุด เมื่อมีเฉลย